# 05 — Exploración no supervisada: ¿normal y patológico se separan solos?

**Objetivo:** responder la pregunta de tu directora: *¿qué pasa si usamos
k-means?* Al terminar deberías poder explicar:
- la diferencia entre aprendizaje **supervisado** y **no supervisado**;
- por qué hay que **estandarizar** las features antes de medir distancias;
- qué hace el **PCA** y cómo leer un gráfico de componentes principales;
- cómo funciona **k-means** y cómo se compara su resultado con las etiquetas
  (índice de Rand ajustado, silueta);
- qué nos dice que k-means *no* encuentre las clases.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import StandardScaler

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT))
from src.paths import OUT_DIR

NO_FEATURES = ["file_id", "label", "label_binary"]
SEMILLA = 42


def cargar_tabla(subset, normalizada=False):
    """Features de la tesis. normalizada=True: extraídas con la señal normalizada a pico 1."""
    sufijo = "_peaknorm" if normalizada else ""
    df = pd.read_csv(OUT_DIR / f"{subset.lower()}_features{sufijo}.csv")
    partes = df["file_id"].str.split("_")
    df["genero"], df["foco"] = partes.str[0], partes.str[-1]
    df["caso"] = partes.str[:2].str.join("_")
    return df


tablas = {s: cargar_tabla(s) for s in ["HS", "LS"]}

## 1. Supervisado vs. no supervisado

- **Supervisado** (lo que hace la tesis): el modelo recibe las features **y la
  etiqueta** (normal/patológico) y aprende a predecir la etiqueta. Es como
  estudiar con las respuestas al lado.
- **No supervisado:** el modelo recibe **solo las features**, sin etiqueta, y
  busca estructura por su cuenta: grupos de grabaciones parecidas entre sí.

¿Para qué sirve si ya tenemos las etiquetas? Para preguntarnos: **¿la
diferencia entre normal y patológico es la diferencia más grande que hay en
los datos?** Si al agrupar "a ciegas" los grupos coinciden con las etiquetas,
la respuesta es sí. Si no coinciden, la diferencia existe, pero hay otras
fuentes de variación más grandes que la tapan. Después comparamos los grupos
con las etiquetas, que es la gracia del ejercicio.

## 2. Primero: estandarizar

k-means y PCA trabajan con **distancias** entre grabaciones. Las features
tienen escalas muy distintas:

In [ ]:
print(tablas["HS"][["rms_mean", "zcr_mean", "mfcc_1_mean", "spectral_centroid_mean"]].describe()
      .loc[["mean", "std"]].round(4).to_string())

El centroide se mide en cientos de Hz y el RMS en milésimas. Sin hacer nada,
la distancia entre dos grabaciones estaría dominada casi solo por el
centroide. **Estandarizar** (restar la media y dividir por el desvío de cada
feature) pone a todas en la misma escala: media 0, desvío 1. Es lo mismo que
hace el `StandardScaler` dentro de los pipelines de la tesis.

In [ ]:
def estandarizar(df):
    return StandardScaler().fit_transform(df.drop(columns=NO_FEATURES + ["genero", "foco", "caso"]))

## 3. PCA: mirar 105 dimensiones en 2

No podemos graficar 105 features. El **análisis de componentes principales
(PCA)** busca nuevas direcciones en el espacio de las features:
- la **componente 1 (PC1)** es la dirección en la que las grabaciones más
  difieren entre sí (máxima varianza);
- la **PC2** es la siguiente, perpendicular a la primera; y así.

Cada componente es una combinación de las 105 features originales. Si las
primeras dos componentes capturan buena parte de la variación, el gráfico
PC1 vs. PC2 es un buen "resumen visual" de los datos.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for subset, df in tablas.items():
    pca = PCA().fit(estandarizar(df))
    acum = np.cumsum(pca.explained_variance_ratio_)
    ax.plot(np.arange(1, len(acum) + 1), 100 * acum, "o-", ms=3, label=subset)
    print(f"{subset}: PC1 explica {100 * pca.explained_variance_ratio_[0]:.0f} %, "
          f"PC1+PC2 {100 * acum[1]:.0f} %, hacen falta {np.argmax(acum >= 0.9) + 1} componentes para el 90 %")
ax.set(xlabel="Cantidad de componentes", ylabel="% de varianza explicada (acumulada)", xlim=(0, 30),
       title="¿Cuántas direcciones hacen falta para describir los datos?")
ax.legend()
fig.tight_layout()

Ahora el gráfico PC1 vs. PC2, coloreado de dos formas: por **etiqueta** y por
**tipo de sonido**.

In [ ]:
def grafico_pca(df, titulo):
    Z = PCA(2).fit_transform(estandarizar(df))
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    for clase, color in [(0, "tab:blue"), (1, "tab:orange")]:
        m = df.label_binary == clase
        axes[0].scatter(Z[m, 0], Z[m, 1], c=color, alpha=0.7, label="Normal" if clase == 0 else "Patológico")
    axes[0].set(title=f"{titulo} — por etiqueta", xlabel="PC1", ylabel="PC2")
    axes[0].legend()
    for tipo in df.label.unique():
        m = (df.label == tipo).values
        axes[1].scatter(Z[m, 0], Z[m, 1], alpha=0.8, label=tipo)
    axes[1].set(title=f"{titulo} — por tipo de sonido", xlabel="PC1", ylabel="PC2")
    axes[1].legend(fontsize=7, loc="best")
    fig.tight_layout()
    return Z


Z_hs = grafico_pca(tablas["HS"], "Cardíacos")
Z_ls = grafico_pca(tablas["LS"], "Pulmonares")

Preguntas para mirar los gráficos:
- ¿Los normales (azul) forman un grupo aparte, o están mezclados con los
  patológicos?
- ¿Hay tipos de sonido que se agrupen entre sí?

### ¿Qué representa la PC1?

Si la dirección de mayor variación de los datos fuera "normal vs.
patológico", sería ideal. Veamos con qué se relaciona. Calculamos la
correlación de la PC1 con el volumen (`rms_mean`):

In [ ]:
for subset, Z in [("HS", Z_hs), ("LS", Z_ls)]:
    r = np.corrcoef(Z[:, 0], tablas[subset]["rms_mean"])[0, 1]
    print(f"{subset}: correlación entre PC1 y el volumen = {r:+.2f}")

En pulmonares, **la dirección en la que más varían las grabaciones es
básicamente el volumen**. Es lo que anticipaban las notebooks 02 y 03: varias
features siguen al volumen, así que el volumen es la "fuente de variación"
más grande. ¿Qué pasa si usamos las features extraídas con la señal
normalizada (pico = 1)?

In [ ]:
for subset in ["HS", "LS"]:
    df_n = cargar_tabla(subset, normalizada=True)
    Z_n = PCA(2).fit_transform(estandarizar(df_n))
    r = np.corrcoef(Z_n[:, 0], tablas[subset]["rms_mean"])[0, 1]
    print(f"{subset} con señal normalizada: correlación entre PC1 y el volumen original = {r:+.2f}")

Resultado distinto en cada subconjunto:
- **Cardíacos:** al normalizar, la PC1 deja de estar relacionada con el volumen.
- **Pulmonares:** aun con la señal normalizada, la PC1 sigue muy relacionada
  con el volumen **original**.

¿Cómo puede ser, si normalizamos? Normalizar divide la señal por un número:
elimina el **nivel**, pero no cambia la **forma** del espectro. Y una
grabación hecha con menos volumen no es solo "la misma, más baja": en
proporción tiene más ruido de fondo, y eso cambia la forma de su espectro.
Lo comprobamos: ¿qué features de la tabla normalizada siguen asociadas al
volumen original?

In [ ]:
for subset in ["HS", "LS"]:
    df_n = cargar_tabla(subset, normalizada=True)
    volumen_original = tablas[subset].set_index("file_id").loc[df_n.file_id, "rms_mean"].values
    X_n = df_n.drop(columns=NO_FEATURES + ["genero", "foco", "caso"])
    corr = X_n.corrwith(pd.Series(volumen_original, index=X_n.index), method="spearman").abs()
    print(f"\n{subset} (señal normalizada): features más asociadas al volumen ORIGINAL")
    print(corr.sort_values(ascending=False).head(5).round(2).to_string())

En pulmonares, el contraste espectral de la banda 5 tiene una correlación de
~0,99 con el volumen original **aun después de normalizar**. La conclusión
importante:

> Lo que distingue a las grabaciones más bajas no es solo el nivel, sino una
> **condición de grabación** (más ruido de fondo en proporción) que queda
> impresa en la forma del espectro. Normalizar la amplitud **no alcanza** para
> quitarle esa pista al modelo.

Esto corrige lo que dijimos en las notebooks 02 y 03 ("normalizar elimina la
pista del volumen"): la elimina solo en parte. En la notebook de comparación
(11) vamos a ver cuánto pesa en el desempeño.

## 4. k-means

k-means arma **k grupos** (*clusters*) de grabaciones parecidas. El algoritmo:
1. elige k puntos al azar como "centros" iniciales;
2. asigna cada grabación al centro más cercano;
3. mueve cada centro al promedio de las grabaciones que le tocaron;
4. repite 2 y 3 hasta que los centros dejan de moverse.

Como el resultado depende del arranque al azar, se corre varias veces
(`n_init=20`) y se queda con la mejor. El número de grupos **k lo elegimos
nosotras**. Empezamos con k = 2, la cantidad de clases.

In [ ]:
for subset, df in tablas.items():
    grupos = KMeans(n_clusters=2, n_init=20, random_state=SEMILLA).fit_predict(estandarizar(df))
    tabla = pd.crosstab(pd.Series(grupos, name="grupo k-means"),
                        df.label_binary.map({0: "Normal", 1: "Patológico"}).rename("etiqueta"))
    print(f"\n{subset}:\n{tabla.to_string()}")

Los números de grupo (0 y 1) son arbitrarios: k-means no sabe qué es "normal".
Lo que importa es si un grupo concentra a los normales y el otro a los
patológicos.

### Medir la coincidencia: el índice de Rand ajustado (ARI)

El **ARI** compara dos formas de agrupar las mismas grabaciones (acá: los
grupos de k-means y las etiquetas reales):
- **ARI = 1:** coinciden perfectamente;
- **ARI ≈ 0:** coinciden lo mismo que si los grupos se hubieran sorteado al azar;
- no importa cómo se numeren los grupos.

In [ ]:
for subset, df in tablas.items():
    grupos = KMeans(n_clusters=2, n_init=20, random_state=SEMILLA).fit_predict(estandarizar(df))
    print(f"{subset}: ARI entre k-means (k=2) y la etiqueta = {adjusted_rand_score(df.label_binary, grupos):+.2f}")

## 5. ¿Hay grupos "naturales" en los datos? La silueta

Quizás los datos sí forman grupos, pero no los de la etiqueta. La **silueta**
mide qué tan bien separados están los grupos que armó k-means, para cada
grabación: compara la distancia a los de su propio grupo con la distancia al
grupo vecino.
- **cerca de 1:** grupos compactos y bien separados;
- **cerca de 0:** los grupos se tocan, no hay una separación clara;
- **negativa:** la grabación parece estar en el grupo equivocado.

Probamos varios k y, además del ARI con la etiqueta, el ARI con otras cosas
que podrían estar organizando los datos: el **tipo de sonido**, el **caso**
(género + tipo), el **foco** de auscultación y el **género**.

In [ ]:
for subset, df in tablas.items():
    X = estandarizar(df)
    filas = []
    for k in [2, 3, 4, 6, 8, 10]:
        g = KMeans(n_clusters=k, n_init=20, random_state=SEMILLA).fit_predict(X)
        filas.append({"k": k, "silueta": silhouette_score(X, g),
                      **{f"ARI {col}": adjusted_rand_score(df[c], g)
                         for col, c in [("etiqueta", "label_binary"), ("tipo", "label"),
                                        ("caso", "caso"), ("foco", "foco"), ("género", "genero")]}})
    print(f"\n{subset}:")
    print(pd.DataFrame(filas).round(2).to_string(index=False))

Preguntas para mirar la tabla:
- ¿La silueta llega a valores altos para algún k? ¿Qué te dice eso?
- ¿Algún ARI se aleja claramente de 0? ¿Con qué variable?

## 6. ¿Qué nos dice que k-means no encuentre las clases?

Hay que interpretarlo con cuidado, porque **no** significa que normal y
patológico sean indistinguibles: los modelos supervisados de la tesis sí los
separan (sobre todo en pulmonares). Significa otras tres cosas:

1. **La diferencia entre clases no es la mayor fuente de variación.** Entre 50
   grabaciones hay muchas diferencias (volumen, foco, tipo de patología, el
   artefacto de 146 Hz...). k-means agrupa por lo que más varía; la etiqueta
   vive en unas pocas direcciones que quedan "tapadas".
2. **Muchas features, pocas grabaciones (p ≫ n).** Con 105 features, la
   distancia entre dos grabaciones suma diferencias en todas, la mayoría
   irrelevantes para la etiqueta. Las pocas features informativas pesan poco
   en esa suma. Un modelo supervisado, en cambio, aprende **qué features
   mirar** porque tiene la etiqueta.
3. **"Patológico" no es un grupo acústico.** Mezcla soplos, ruidos de baja
   frecuencia y arritmias (notebook 03, sección 10), que no se parecen entre
   sí. No hay razón para que formen un solo "montón".

**Para la tesis y para tu directora:** el análisis no supervisado es un buen
**análisis exploratorio**: muestra que la etiqueta no es la estructura
dominante de los datos, y que en pulmonares esa estructura dominante es una
condición de grabación ligada al volumen, que normalizar la amplitud no
elimina del todo. Por eso el aprendizaje supervisado es necesario, y por eso
hay que controlar esa condición de grabación. No es un clasificador
alternativo.

## 7. Tu turno

Probá cambiar el subconjunto, usar las features normalizadas o cambiar k.
¿Cambian los grupos? ¿Coinciden más con alguna variable?

In [ ]:
SUBSET, NORMALIZADA, K = "LS", True, 6

df = cargar_tabla(SUBSET, normalizada=NORMALIZADA)
X = estandarizar(df)
g = KMeans(n_clusters=K, n_init=20, random_state=SEMILLA).fit_predict(X)
print(f"Silueta: {silhouette_score(X, g):.2f}")
print(pd.crosstab(pd.Series(g, name="grupo"), df["label"]).to_string())

## Preguntas de control

1. ¿Qué diferencia hay entre aprendizaje supervisado y no supervisado? Si ya
   tenemos las etiquetas, ¿para qué sirve k-means acá?
2. ¿Por qué hay que estandarizar antes de PCA o k-means?
3. ¿Qué es la PC1? En pulmonares, ¿con qué está relacionada? ¿Por qué
   normalizar la amplitud no alcanza para quitar esa relación?
4. Explicá los pasos de k-means. ¿Por qué se corre varias veces?
5. ¿Qué miden el ARI y la silueta? ¿Qué valores obtuvimos y qué significan?
6. Si k-means no separa normales de patológicos, ¿por qué un modelo
   supervisado sí puede? Da al menos dos razones.